# Exception Handling

A GIS script often works with data it did not create: coordinates typed by a field crew, station names copied from a spreadsheet, values delivered by another agency. One bad value, such as a coordinate entered as `"N/A"`, raises an exception and stops the whole script. Exception handling lets a script respond to an exception instead of stopping: it can report the problem, skip the bad value and keep going, or stop with a clearer message. ArcPy geoprocessing tools raise exceptions when they fail, so the same techniques apply when you run geoprocessing tools from Python.

🎯 What You'll Learn

* Catch an exception with `try` and `except`
* Handle exceptions inside a loop so that one bad value does not stop the loop
* Predict which exceptions a line of code can raise
* Respond to different exception types with multiple `except` clauses
* Run code only when nothing failed with `else`, and every time with `finally`
* Recognize how a function raises an exception with `raise`, and read a traceback with more than one frame

Work through each numbered section: a demo shows how something works, and most sections end with a ✏️ **Practice** prompt that gives you a chance to try it yourself.

### ⚙️ Before you start: select the kernel

The **kernel** is the program that runs the Python code in this notebook. Before running any cells, make sure the notebook is using the ArcGIS Pro Python environment. An **environment** is a copy of Python plus the add-on tools installed with it.

Check the kernel name in the top-right corner of the notebook. It should read `arcgispro-py3`, and may include a Python version, for example:
`arcgispro-py3 (Python 3.13.13)`

If it shows a different name or says **Select Kernel**:

1. Click the kernel name (or **Select Kernel**) in the top-right corner
2. Select `arcgispro-py3`
3. If it is not listed, refer back to `jupyter-notebook-basics.ipynb` in `M1-getting-started-with-python\code-alongs`

---

### 1. try and except

Let us revisit an exception from the Types of Errors notebook in Module 1.

`float()` raised a `ValueError` because it could not convert `"N/A"` to a number. The exception stopped the cell at that line, and every line below it was skipped. Run the cell below and read its traceback.

In [ ]:
# This cell raises an exception on purpose
y_text = "N/A"  # a missing value, stored as text
y_value = float(y_text)  # raises a ValueError
print(f"y_value: {y_value}")  # never runs
print("Finished converting")  # never runs

The cell stopped at line 3, so neither `print()` ran.

To **handle** an exception means to write code that runs when the exception is raised, so that the program responds instead of stopping. Python handles exceptions with a `try` statement, which has two parts:

* The **try block** is the indented code under `try:`. Python runs it line by line, as usual.
* The **except clause** starts with the keyword `except` and a colon. Its indented code runs only if a line in the try block raises an exception.

You will often see this described as "try and catch," because **catching** an exception is another word for handling it. Python's keyword is `except`, not `catch`.

In [ ]:
y_text = "N/A"  # a missing value, stored as text
print(f"y_text: {y_text}")

# Python runs the try block line by line
try:
    y_value = float(y_text)  # raises an exception
    print(f"\ty_value: {y_value}")  # skipped, because the line above failed
# The except clause runs only if the try block raises an exception
except:
    print(f"\tCould not convert '{y_text}' to a number")

# Lines after the try statement run whether or not an exception was raised
print("Finished converting")

This time the exception was **caught**: instead of stopping the cell with a traceback, Python skipped the rest of the try block and ran the except clause. After the except clause, the program continued with the next line after the try statement, so `Finished converting` printed.

The cell below has the same code, with `y_text` changed to a value that `float()` can convert.

In [ ]:
y_text = "4905101.303"  # a valid y coordinate, stored as text
print(f"y_text: {y_text}")

# Python runs the try block line by line
try:
    y_value = float(y_text)  # converts without an exception
    print(f"\ty_value: {y_value}")  # runs, because the line above worked
# The except clause is skipped, because no exception was raised
except:
    print(f"\tCould not convert '{y_text}' to a number")

# Lines after the try statement run whether or not an exception was raised
print("Finished converting")

The try block finished without an exception, so every line in it ran and the except clause was skipped.

An `except` with nothing after it catches every exception, whatever caused it. Section 2 shows why that is a problem, and how to catch only the exceptions you expect.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 2. Exception types

The except clause in Section 1 caught every exception. That sounds safe, but it also catches exceptions you did not expect, including mistakes in your own code. In the cell below, the variable name in the try block is misspelled.

In [ ]:
y_text = "N/A"  # a missing value, stored as text
print(f"y_text: {y_text}")

try:
    # The variable name is misspelled: y_txt instead of y_text
    y_value = float(y_txt)
    print(f"\ty_value: {y_value}")
# A bare except catches every exception, including the misspelling
except:
    print(f"\tCould not convert '{y_text}' to a number")

The message says the value could not be converted, but that is not what went wrong. `float()` never ran: Python could not find a variable named `y_txt` and raised a `NameError`. The except clause caught it and printed a message that hides the real problem. Change `y_text` to `"4905101.303"` and rerun the cell: the message is the same, even though the value is valid.

Every exception has an **exception type**, the name on the last line of its traceback, such as `ValueError` or `NameError`, which you saw in the Types of Errors notebook. Writing a type after `except`, such as `except ValueError:`, makes the except clause catch only exceptions of that type. Any other exception stops the cell with a traceback, as if there were no try statement.

In [ ]:
# This cell raises an exception on purpose
y_text = "N/A"  # a missing value, stored as text
print(f"y_text: {y_text}")

try:
    # The variable name is still misspelled
    y_value = float(y_txt)
    print(f"\ty_value: {y_value}")
# Catches only a ValueError, so the NameError is not caught
except ValueError:
    print(f"\tCould not convert '{y_text}' to a number")

Now the traceback names the real problem: `NameError: name 'y_txt' is not defined`. A misspelled variable name is a mistake in your own code, so fix it instead of handling it. Handle exceptions caused by things your code cannot control, such as values typed by someone else.

The cell below fixes the misspelling. The `ValueError` from `"N/A"` is caught.

In [ ]:
y_text = "N/A"  # a missing value, stored as text
print(f"y_text: {y_text}")

try:
    y_value = float(y_text)  # raises a ValueError
    print(f"\ty_value: {y_value}")
# Catches only a ValueError
except ValueError:
    print(f"\tCould not convert '{y_text}' to a number")

How do you know which exception type to write after `except`? Here, the traceback from the first cell in Section 1 named it: `ValueError`. Section 4 shows how to predict exception types before you run the code.

The except clause above printed our own message, and Python's original message, `could not convert string to float: 'N/A'`, was lost. That original message is often the most useful clue about what went wrong. When Python raises an exception, it creates an **exception object**, a value that stores the exception type and its message. Adding `as` and a variable name to the except clause, such as `except ValueError as e`, stores the exception object in that variable. Printing the variable prints the message, and `type()` shows the exception type. The name `e` is a common convention, but any variable name works.

In [ ]:
# An x coordinate stored as text, with a comma that float() cannot read
x_text = "610,124.0815"
print(f"x_text: {x_text}")

try:
    x_value = float(x_text)  # raises a ValueError
    print(f"\tx_value: {x_value}")
# as e stores the exception object in the variable e
except ValueError as e:
    print(f"\tMessage: {e}")  # printing e prints the message
    print(f"\tType: {type(e)}")  # type() shows the exception type

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 2a] Run this cell and read the last line of the traceback. Then edit
# this cell: put the lookup and the print() in a try block, and add an
# except clause for the exception type named in the traceback. Use as to
# store the exception object, and in the except clause, print the message
# and the type of the exception object, labeling each value.

# 2b] Change requested_w_id to an ID between 1 and 9 and rerun the cell.
# Confirm that the except clause was skipped.

# List stores the pour point names in W_ID order (W_ID 1 is index 0)
station_names = [
    "Elk 1", "Bear Butte 1", "Bear Butte 2", "Bear Butte 3", "Elk 2",
    "Whitewood 1", "Bear Butte 4", "Whitewood 2", "Whitewood 3",
]

# W_ID of the pour point to look up, requested by someone else
requested_w_id = 12

# Look up the station name. The index is one less than the W_ID
station_name = station_names[requested_w_id - 1]

# Print the name of the station with the requested W_ID
print(f"station_name: {station_name}")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 3. Handling exceptions in a loop

In the Loops notebook, a `for` loop ran the same code once for each item in a list. If one item raises an exception, the loop stops at that item, and the items after it are never processed. Run the cell below.

In [ ]:
# This cell raises an exception on purpose

# List stores the Y coordinates of four pour points as text. The third
# value is missing, typed as "N/A"
y_coords_text = ["4905101.303", "4913979.832", "N/A", "4910997.032"]
print(f"y_coords_text: {y_coords_text}")

# Loop over the list, one Y coordinate at a time
for one_y_text in y_coords_text:
    # Convert the text to a float. Raises a ValueError on "N/A", which
    # stops the loop
    y_value = float(one_y_text)

    # Print the converted value. Does not run for "N/A" or any item after it
    print(f"\tConverted {y_value}")

The loop converted the first two values, then stopped at `"N/A"`. The fourth value was never converted, even though nothing was wrong with it.

Placing the try statement *inside* the loop means it runs once for each item. When one item raises an exception, the except clause handles it, and the loop moves on to the next item. The demo below also stores the values that could not be converted in a list, so the program can report them in a summary after the loop.

In [ ]:
# List stores the Y coordinates of four pour points as text. The third
# value is missing, typed as "N/A"
y_coords_text = ["4905101.303", "4913979.832", "N/A", "4910997.032"]
print(f"y_coords_text: {y_coords_text}")

y_values = []  # stores the values that were converted
failed_values = []  # stores the values that could not be converted

for index, one_y_text in enumerate(y_coords_text, start=1):
    # The try statement is inside the loop, so it runs once for each item
    try:
        # Convert the text in one_y_text to a float
        y_value = float(one_y_text)
        # Append the float y_value to the y_values list
        y_values.append(y_value)
        print(f"\tItem {index}: converted {y_value}")
    except ValueError as e:
        # Append the text in one_y_text to failed_values list
        failed_values.append(one_y_text)
        print(f"\tItem {index}: skipped. {e}")

# After the loop, summarize the results
print(f"Converted {len(y_values)} of {len(y_coords_text)} values")
print(f"failed_values: {failed_values}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# Run this cell once to store the two lists below. Then write your code
# below the lists.

# List stores the pour point names in W_ID order (W_ID 1 is index 0)
station_names = [
    "Elk 1", "Bear Butte 1", "Bear Butte 2", "Bear Butte 3", "Elk 2",
    "Whitewood 1", "Bear Butte 4", "Whitewood 2", "Whitewood 3",
]

# W_IDs of the pour points to look up, requested by someone else
requested_w_ids = [3, 15, 7, 10, 1]


# 3a] Create two empty lists: one for the station names that were found
# and one for the W_IDs that could not be found.


# 3b] Write a for loop over requested_w_ids with a try statement inside
# it. In the try block, look up the station name for each W_ID (the index
# is one less than the W_ID) and append it to the correct list from 3a.
# Add an except clause for the exception type from Section 2's practice,
# use as to store the exception object, and append the W_ID to the other
# list. Print a message for each W_ID, with the exception's message for
# the ones that could not be found.


# 3c] After the loop, print how many W_IDs were found out of the total,
# and print the list of W_IDs that could not be found.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 4. Predicting exceptions

In Sections 1 through 3, you knew `"N/A"` was in the data before you ran the code. When you write a script, you usually do not know every value it will receive, and you cannot test every possible value. Instead, you predict what could go wrong before you run the code, by asking three questions about each line that uses a value:

1. **Where does the value come from?** A value you typed into your own code is known. A value that comes from somewhere else, such as data typed by a field crew, is not.
2. **What else could it be?** Think of the realistic ways the value could differ from what you expect: missing (`"N/A"` or an empty string `""`), formatted differently (commas, units, or extra spaces), or not there at all (a name that is not a key in a dictionary).
3. **What will this line do with each of those values?** Use the exception types from the Types of Errors notebook. If the data type is right but the value cannot be used, expect a `ValueError`. If the data type is wrong, expect a `TypeError`.

In the demo below, the comments are predictions written before the cell was run. The loop checks each one.

In [ ]:
# Y coordinates are typed by a field crew, so list the ways a value could
# be typed. Each comment is a prediction, written before running the cell
possible_y_values = [
    "4905101.303",  # expected format: converts
    "4,905,101.303",  # commas: ValueError
    "4905101.303 m",  # units: ValueError
    "N/A",  # missing, typed as text: ValueError
    "",  # missing, left empty: ValueError
    " 4905101.303 ",  # extra spaces: predicted ValueError
]

# Check each prediction
for one_value in possible_y_values:
    try:
        y_value = float(one_value)
        print(f"\t'{one_value}' converted to {y_value}")
    except ValueError as e:
        print(f"\t'{one_value}' raised a ValueError: {e}")

Every prediction was right except the last one: `float()` ignores spaces at the start and end of text, so `" 4905101.303 "` converted. A wrong prediction like this is not a problem. It only means a value you planned to handle did not need handling.

The opposite case is an exception you did not predict. It stops the cell with a traceback that names it, and you decide then whether to fix the code or handle it. You do not need to predict every exception: handle the problems you expect from data you cannot control. Trying to handle every possible exception leads to the catch-all pattern in Section 5, which hides problems instead of reporting them.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 5. Multiple except clauses

Asking the three questions from Section 4 often turns up more than one exception type for the same few lines of code. 

For example, looking up a station's Y coordinate in a dictionary and converting it can fail in three ways:
* The station name is not a key in the dictionary. Looking up a key that a dictionary does not contain raises a `KeyError`.
* The value is text that does not look like a number, which raises a `ValueError`.
* No value was recorded at all. The demo stores this as `None`, Python's value for "no value," which you saw in the Functions notebook. `float(None)` raises a `TypeError`, because `None` is not text or a number.

A try statement can have several except clauses, one for each exception type, so that each problem gets its own response. Python checks the except clauses from top to bottom and runs only the **first** one that matches the exception type. The rest are skipped.

`Exception` is a type that matches almost every exception, so `except Exception` is called a **catch-all**. Because Python runs only the first match, a catch-all goes last, where it handles only the exceptions that no clause above it matched.

In [ ]:
# Dictionary stores the Y coordinate of each pour point, as text
pour_point_y = {
    "Elk 1": "4905101.303",
    "Bear Butte 1": "4913979.832",
    "Bear Butte 4": "N/A",  # a missing value, typed as text
    "Whitewood 3": None,  # no value was recorded
}

# Change station_name and rerun the cell to reach each except clause:
#   "Elk 1" (no exception), "Elk 3" (KeyError),
#   "Bear Butte 4" (ValueError), "Whitewood 3" (TypeError, caught by the
#   catch-all)
station_name = "Elk 3"
print(f"station_name: {station_name}")

try:
    # Look up the Y coordinate. Raises a KeyError if the station name is
    # not a key in the dictionary
    station_y_text = pour_point_y[station_name]

    # Convert the Y coordinate to a float. Raises a ValueError for text
    # that is not a number, or a TypeError for None
    y_value = float(station_y_text)

    # Runs only if both lines above worked
    print(f"\ty_value: {y_value}")
# Python checks each except clause from top to bottom and runs only the
# first one that matches
except KeyError as e:
    # The message of a KeyError is the missing key, in quotes
    print(f"\tStation not found: {e}")
except ValueError as e:
    print(f"\tBad coordinate value: {e}")
except Exception as e:
    # The catch-all matches almost every exception type, so it goes last.
    # It prints the type and message, so the problem is still reported
    print(f"\tUnexpected {type(e)}: {e}")

You could also avoid the `KeyError` by checking first, with `if station_name in pour_point_y:`. Both approaches are common. A try statement is useful when several different problems can happen in a few lines, as in the cell above.

AI-generated code often uses a catch-all that does nothing, written with `pass`, a keyword that does nothing at all. This hides every problem, including bugs in your own code such as a misspelled variable name: the script keeps running and gives no sign that anything went wrong. Older code also uses a **bare except**, `except:` with no exception type, which catches every exception in the same way. Recognize both patterns, but do not write them. When you see them in AI-generated code, replace them with the specific exception types you expect.

In [ ]:
# Recognize these patterns, but do not write them

# A catch-all that silently ignores every exception
# try:
#     y_value = float(pour_point_y[station_name])
# except Exception:
#     pass

# A bare except, which also catches every exception
# try:
#     y_value = float(pour_point_y[station_name])
# except:
#     pass

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 5a] Create a dictionary of at least three places you have visited, with
# each place name as a key and its elevation, stored as text, as the value.
# Make at least one value something float() cannot convert. Print the
# dictionary.


# 5b] Create a variable to store a place name. Write a try statement that
# looks up the place in your dictionary from 5a and converts its elevation
# with float(). Add an except clause for KeyError and one for ValueError,
# each printing its own message with the exception's message. Add a
# catch-all except clause last that prints the exception's type and
# message.


# 5c] Rerun this cell three times, changing your variable from 5b to: a
# place in your dictionary with a valid elevation, a place that is not in
# your dictionary, and the place whose elevation cannot be converted.
# Confirm which except clause runs each time.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 6. else and finally

In Section 5, the `print()` of the converted value is inside the try block. Any line in a try block can be handled by the except clauses, so a try block should hold only the lines you expect to raise an exception. Code that should run only when those lines succeed belongs somewhere else.

A try statement can have two more clauses, written after the except clauses, in this order:

* The **else clause** runs only if the try block raised no exception.
* The **finally clause** runs every time, whether or not an exception was raised. It is useful for steps that must always happen, such as a message that a step is complete.

In [ ]:
# We will use variable pour_point_y from Section 5

# Change station_name to "Elk 3" and rerun the cell to see the except
# clause run instead of the else clause
station_name = "Bear Butte 1"
print(f"station_name: {station_name}")

try:
    # Only the lines that can raise an exception go in the try block

    # Look up the Y coordinate. Raises a KeyError if the station name is
    # not a key in the dictionary
    station_y_text = pour_point_y[station_name]

    # Convert the Y coordinate to a float. Raises a ValueError for text
    # that is not a number
    y_value = float(station_y_text)
except KeyError as e:
    # Runs only if the lookup raised a KeyError
    print(f"\tStation not found: {e}")
except ValueError as e:
    # Runs only if the conversion raised a ValueError
    print(f"\tBad coordinate value: {e}")
else:
    # Runs only if the try block raised no exception
    print(f"\ty_value: {y_value}")
finally:
    # Runs every time, whether or not an exception was raised
    print(f"\tLookup complete for {station_name}")

This try statement has no catch-all. Change `station_name` to `"Whitewood 3"` and rerun the cell: no except clause matches the `TypeError`, so the finally clause runs and then the exception stops the cell with a traceback. A finally clause runs even when an exception is not handled.

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 6a] Copy your try statement from 5b into this cell. Move the print() of
# the converted elevation out of the try block and into an else clause.


# 6b] Add a finally clause that prints a message saying the lookup is
# complete, including the place name.


# 6c] Rerun this cell with a place in your dictionary with a valid
# elevation, then with a place that is not in your dictionary. Confirm
# when the else clause runs and when the finally clause runs.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 7. raise

So far, every exception was raised by Python itself. In the Functions notebook, `create_linear_unit("Meters", 5000)` returned `"Meters 5000"`, a wrong result, without any error. A function can check its own arguments and raise an exception when one is not valid.

To **raise** an exception means to create one yourself. You write the keyword `raise`, followed by an exception type and a message in parentheses, such as `raise ValueError("message")`. Like `return`, `raise` ends the function immediately, but instead of sending back a value, it sends back an exception. The code that called the function can handle it with a try statement, or let it stop the program with your message.

Choose the exception type the same way you predicted types in Section 4. In the demo below, the unit is a string, the right data type, but its value cannot be used, so the function raises a `ValueError`.

Raising is better than printing a warning and returning nothing. A function without `return` gives back `None`, so the code that called it would keep going with a value it cannot use.

A function's docstring can list the exceptions it raises under a `Raises:` heading, the same way `Args:` and `Returns:` describe its parameters and return value.

In [ ]:
# Define create_linear_unit again, this time checking the unit argument
def create_linear_unit(distance, unit):
    """Return a linear unit string, such as "5000 Meters".

    Args:
        distance: The distance, as a number.
        unit: The unit of the distance: "Meters", "Kilometers", "Feet",
            or "Miles".

    Returns:
        The distance and unit as one string.

    Raises:
        ValueError: If unit is not one of the accepted units.
    """
    # List stores the units this function accepts
    valid_units = ["Meters", "Kilometers", "Feet", "Miles"]

    # Check the argument before using it
    if unit not in valid_units:
        # raise ends the function and sends back a ValueError
        raise ValueError(f"'{unit}' is not one of {valid_units}")

    # These lines run only if the unit is valid
    linear_unit = f"{distance} {unit}"
    return linear_unit

In [ ]:
# Calling the function with a valid unit, so the function returns a value
buffer_distance = create_linear_unit(500, "Meters")
print(f"buffer_distance: {buffer_distance}")

In [ ]:
# This cell raises an exception on purpose
# Calling the function with an invalid unit.
buffer_distance = create_linear_unit(500, "Parsecs")
print(f"buffer_distance: {buffer_distance}")

The last line of the traceback shows the exception type and the message written in the function.

This traceback has two parts, called **frames**, one for each place the code was running when the exception was raised:

* The first frame points to the line in this cell that called the function.
* The second frame shows the name of the function, `create_linear_unit`, and points to the `raise` line inside it.

Read it bottom-up as before: the last line tells you what went wrong, and the frames above it show the path Python took to get there, from your call into the function. Code that calls functions written by others, such as ArcPy tools, can produce tracebacks with many frames.

The code that calls the function can handle the exception like any other.

In [ ]:
# Change requested_unit to "Feet" and rerun the cell to see the else clause
requested_unit = "Parsecs"
print(f"requested_unit: {requested_unit}")

try:
    buffer_distance = create_linear_unit(500, requested_unit)
except ValueError as e:
    # e stores the message written in the function's raise line
    print(f"\tCould not create the linear unit: {e}")
else:
    print(f"\tbuffer_distance: {buffer_distance}")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this notebook, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

*End of notebook.*